# From exploratory model to preprint

This notebook is a presentation-oriented walkthrough of the simulations behind the CA1 plasticity preprint. It reconstructs the relevant plots from saved result artifacts, so running it does **not** repeat the expensive simulations or alter their data.

The central proposal is deliberately narrow: a CA3-like key selects an association, while rapid CA3–CA1 plasticity expresses the recalled content in a pretrained CA1 representational basis that remains interpretable to a fixed downstream decoder.

## The argument in one pass

1. **Readout compatibility:** storing an informative target is insufficient if its coordinates are incompatible with the established decoder.
2. **Cue-dependent reconfiguration:** exchanging cue identities changes CA1 tuning beyond the drift caused by continued learning.
3. **Key reliability under corruption:** a structured CA3-like key supports recall from degraded inputs better than controls that destroy item-specific key structure.
4. **Complementary EC contributions:** selective LEC and MEC corruption preferentially affects cue and spatial recall, respectively.
5. **Plasticity-rule trade-off:** bounded error correction improves clean target discrimination, whereas direct instructed writing retains a more graded similarity signal under severe corruption.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, SVG, display

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src/experiments/preprint').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the kam repository.')

ROOT = find_repository_root(Path.cwd().resolve())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RESULTS = ROOT / 'results/preprint/v1'
ARTICLE_FIGURES = ROOT / 'article/figures/preprint'
PLOT_CACHE = Path(tempfile.mkdtemp(prefix='kam-preprint-walkthrough-'))

from experiments.preprint.artifacts import load_arrays
from experiments.preprint.figures.figure_2_compatibility import build as build_compatibility
from experiments.preprint.figures.figure_3_cue_remapping import build as build_tuning_distribution
from experiments.preprint.figures.figure_4_completion import build as build_completion
from experiments.preprint.figures.figure_5_plasticity_ablation import build as build_rule_ablation
from experiments.preprint.figures.figure_cue_swap_control import build as build_swap_control
from experiments.preprint_figure_3_experiment import build as build_degradation_figure

# The production figure modules use the non-interactive Agg backend when
# saving files. Restore Jupyter's inline backend for plots made in cells.
get_ipython().run_line_magic('matplotlib', 'inline')

def show_built_figure(builder, *arguments, filename: str, width: int = 1000):
    output = PLOT_CACHE / filename
    builder(*arguments, output)
    display(Image(filename=str(output), width=width))
    return output

print(f'Repository: {ROOT}')
print(f'Temporary plot cache: {PLOT_CACHE}')

## Reproducibility and data lineage

Each final runner saved a resolved configuration, compressed numerical arrays, tidy source data, a report, and a manifest. The root seed—not a unit, position, memory, or corruption mask—is the unit of computational replication. The table below is a quick check that the core artifacts are available.

In [ ]:
artifact_names = [
    'compatibility',
    'cue_remapping',
    'cue_swap_control',
    'completion',
    'plasticity_ablation',
]

rows = []
for name in artifact_names:
    artifact = RESULTS / name
    report_path = artifact / 'report.json'
    report = json.loads(report_path.read_text())
    arrays = load_arrays(artifact)
    rows.append((
        name,
        len(arrays.get('root_seeds', [])),
        report.get('scientific_digest', '')[:12],
        (artifact / 'source_data.csv').exists(),
    ))

print(f"{'artifact':<24} {'seeds':>5}  {'digest':<12}  source data")
for name, seeds, digest, has_source in rows:
    print(f'{name:<24} {seeds:>5}  {digest:<12}  {has_source}')

## 1. Circuit abstraction and learning stages

Figure 1 separates three operations that should not be conflated:

- **Pretraining** learns an EC–CA1–EC representational basis and decoder.
- **Rapid storage** associates a fixed CA3-like retrieval key with the instructed CA1 target.
- **Frozen recall** tests whether CA1 output remains decodable, including under cue changes or corrupted EC input.

In [ ]:
figure_1_svg = ARTICLE_FIGURES / 'figure_1.svg'
if figure_1_svg.exists():
    display(SVG(filename=str(figure_1_svg)))
else:
    display(Image(filename=str(ARTICLE_FIGURES / 'figure_1.png'), width=1100))

### Preliminary evolutionary search and final training

During model development, bounded hyperparameter settings were explored with **CMA-ES** (covariance matrix adaptation evolution strategy). Candidate configurations were scored by reconstruction- or recall-based objectives. This was an engineering outer loop used to locate workable regimes for sparsity, activation gains, CA3-like convergence, and plasticity rate.

The final workflow was:

```text
CMA-ES exploration -> choose one fixed configuration
                      -> train each autoencoder with Adam
                      -> freeze encoder and decoder
                      -> apply local CA3-CA1 plasticity
                      -> evaluate paired conditions over 20 seeds
```

CMA-ES is therefore neither part of recall nor a proposed biological learning mechanism. It was not rerun separately for each reported condition, and the plasticity-rule comparison uses shared rather than separately optimized parameters.

### How the CMA-ES search behaves

The original model-search trajectories were not retained as immutable preprint artifacts. The illustration below therefore runs the repository's actual CMA-ES implementation on a small two-parameter toy objective. It demonstrates the search procedure—not evidence from the biological model. Candidate populations contract and move as the covariance matrix adapts, while the best objective value improves across generations.

In [ ]:
evolution_directory = SRC / 'experiments/evolution'
if str(evolution_directory) not in sys.path:
    sys.path.insert(0, str(evolution_directory))
from experiments.evolution._lib import evolution_run

toy_target = np.array([1.35, -0.85])

def toy_objective(population):
    population = np.asarray(population, dtype=float)
    displacement = population - toy_target
    return (
        displacement[:, 0] ** 2
        + 3.0 * displacement[:, 1] ** 2
        + 0.25 * displacement[:, 0] * displacement[:, 1]
    )

toy_record = evolution_run(
    settings={
        'num_parameters': 2,
        'generations': 24,
        'population_size': 8,
        'direction': 'minimize',
        'verbose': False,
        'disable': True,
        'metric_name': 'toy loss',
        'workers': 1,
    },
    evaluate=toy_objective,
    live_plot=False,
)

figure, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
generations = np.asarray(toy_record['generations']) + 1
axes[0].plot(generations, toy_record['population_mean_fitness'], color='0.55', label='Population mean')
axes[0].plot(generations, toy_record['best_fitness'], 'o-', color='#7b3294', label='Best so far')
axes[0].set(xlabel='Generation', ylabel='Toy objective (lower is better)', yscale='log', title='Illustrative CMA-ES convergence')
axes[0].legend(frameon=False)

x = np.linspace(-2.5, 2.5, 180)
y = np.linspace(-2.5, 2.5, 180)
xx, yy = np.meshgrid(x, y)
surface = toy_objective(np.column_stack((xx.ravel(), yy.ravel()))).reshape(xx.shape)
axes[1].contour(xx, yy, surface, levels=14, colors='0.82', linewidths=0.8)
for generation_index, color, label in ((0, '#d95f02', 'First population'), (7, '#4daf4a', 'Generation 8'), (-1, '#2678b2', 'Final population')):
    population = np.asarray(toy_record['populations'][generation_index])
    axes[1].scatter(population[:, 0], population[:, 1], s=28, alpha=0.75, color=color, label=label)
axes[1].scatter(*toy_target, marker='*', s=180, color='black', label='Optimum')
axes[1].set(xlabel='Candidate parameter 1', ylabel='Candidate parameter 2', title='Adaptation of the search distribution')
axes[1].legend(frameon=False, fontsize=8)
for axis in axes:
    axis.spines[['top', 'right']].set_visible(False)
plt.show()

## 2. Decoder-coordinate compatibility

**Question.** Can a memory retain its information yet fail because it is expressed in coordinates that the fixed decoder does not understand?

The fixed-permutation condition applies the same non-identity permutation to every instructed CA1 target. The matched-decoder condition applies the corresponding permutation to the decoder as an algebraic coordinate control. Restoration by the matched decoder shows that the permuted target remains informative; what fails is compatibility with the unchanged readout.

In [ ]:
compatibility_plot = show_built_figure(
    build_compatibility,
    RESULTS / 'compatibility',
    filename='compatibility.png',
    width=1100,
)

In [ ]:
compatibility_report = json.loads((RESULTS / 'compatibility/report.json').read_text())
for condition, value in compatibility_report['condition_mean_cosine'].items():
    print(f'{condition:<20} {value:.3f}')

print('\nKey contrast: aligned and matched-decoder recall coincide, while fixed permutation fails.')

## 3. Cue-sequence changes and CA1 reconfiguration

Two cues exchange positions after laps 10, 20, and 30. A matched no-swap model shares the seed, pretrained network, initial weights, CA3 wiring, MEC trajectory, lap count, and plasticity parameters. Only the LEC cue schedule differs. The sharp loss of consecutive-lap tuning similarity at swap boundaries isolates cue-sequence change from ordinary learning over time.

In [ ]:
swap_plot = show_built_figure(
    build_swap_control,
    RESULTS / 'cue_swap_control',
    filename='cue_swap_control.png',
    width=900,
)

swap_report = json.loads((RESULTS / 'cue_swap_control/report.json').read_text())
print(f"Swap-boundary similarity: {swap_report['mean_swap_event_similarity']:.3f}")
print(f"Matched no-swap similarity: {swap_report['mean_matched_no_swap_similarity']:.3f}")
print(f"Paired swap effect: {swap_report['mean_paired_swap_effect']:.3f}")

### Heterogeneous tuning after cue exchange

After training, plasticity is frozen and both cue arrangements are probed. Each point below is one CA1 unit. Spatial stability is the cosine similarity between its mean-centered tuning curves; cue modulation is their mean absolute activity difference. The distribution is descriptive: confidence intervals in the manuscript are computed from seed-level summaries, not from the 1,000 unit-level points.

### Tuning curves and their evolution over laps

To make the remapping idea concrete, the next cell selects units by transparent criteria in the same representative seed used for the heat maps: highest cross-context stability, highest cue modulation, and lowest cross-context stability. The upper row compares their frozen tuning curves in contexts A and B. The lower row follows each unit's spatial activity throughout training; horizontal lines mark cue exchanges. These examples are illustrations of points in the population distribution, not additional statistical evidence.

In [ ]:
remapping = load_arrays(RESULTS / 'cue_remapping')
seed_scores = np.column_stack((
    remapping['spatial_stability'].mean(axis=1),
    remapping['cue_modulation'].mean(axis=1),
))
seed_center = np.median(seed_scores, axis=0)
seed_index = int(np.argmin(np.sum((seed_scores - seed_center) ** 2, axis=1)))
stability = remapping['spatial_stability'][seed_index]
modulation = remapping['cue_modulation'][seed_index]
selected_units = [int(np.argmax(stability)), int(np.argmax(modulation)), int(np.argmin(stability))]
selection_labels = ['Highest stability', 'Highest cue modulation', 'Lowest stability']

figure, axes = plt.subplots(2, 3, figsize=(12, 6.8), constrained_layout=True)
positions = np.arange(remapping['probe_ca1'].shape[2])
for column, (unit, label) in enumerate(zip(selected_units, selection_labels)):
    axes[0, column].plot(positions, remapping['probe_ca1'][seed_index, 0, :, unit], color='#2678b2', label='Context A')
    axes[0, column].plot(positions, remapping['probe_ca1'][seed_index, 1, :, unit], color='#d95f02', label='Context B')
    axes[0, column].set(
        title=f'{label}: unit {unit}\nstability={stability[unit]:.2f}, modulation={modulation[unit]:.2f}',
        xlabel='Track position',
        ylabel='CA1 activity',
    )
    if column == 0:
        axes[0, column].legend(frameon=False)

    activity_over_laps = remapping['training_ca1'][seed_index, :, :, unit]
    image = axes[1, column].imshow(activity_over_laps, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=np.quantile(activity_over_laps, 0.98))
    for boundary in (9.5, 19.5, 29.5):
        axes[1, column].axhline(boundary, color='white', linestyle='--', linewidth=0.9, alpha=0.9)
    axes[1, column].set(xlabel='Track position', ylabel='Training lap', title='Tuning evolution')
    figure.colorbar(image, ax=axes[1, column], fraction=0.046, pad=0.03)

for axis in axes[0]:
    axis.spines[['top', 'right']].set_visible(False)
figure.suptitle(f"Representative seed {int(remapping['root_seeds'][seed_index])}: tuning emergence and cue-dependent change", fontsize=14)
plt.show()

In [ ]:
tuning_plot = show_built_figure(
    build_tuning_distribution,
    RESULTS / 'cue_remapping',
    filename='ca1_tuning_distribution.png',
    width=650,
)

remapping = load_arrays(RESULTS / 'cue_remapping')
print(f"Units shown: {remapping['spatial_stability'].size}")
print(f"Mean seed-level spatial stability: {remapping['spatial_stability'].mean(axis=1).mean():.3f}")
print(f"Mean seed-level cue modulation: {remapping['cue_modulation'].mean(axis=1).mean():.3f}")

### Main-text Figure 2

The compatibility experiment, matched temporal control, representative context heat maps, and population tuning distribution are combined into the second main figure. The heat maps visualize examples at population scale; panel E prevents the interpretation from resting only on selected fields.

In [ ]:
display(Image(filename=str(ARTICLE_FIGURES / 'figure_2.png'), width=1150))

## 4. Corrupted inputs and CA3-like key structure

The completion experiment stores clean laps, freezes plasticity, and then masks increasing fractions of EC input. It compares the normal sparse key map with shuffled, dense common-key, and identity controls. The three panels separate output recovery, target identification, and stability of the CA3 key itself.

This is evidence for reliable recall from degraded inputs **conditional on the feedforward key construction**. Because the model has no recurrent CA3 dynamics, it should not be described as a mechanistic demonstration of recurrent CA3 pattern completion.

In [ ]:
completion_plot = show_built_figure(
    build_completion,
    RESULTS / 'completion',
    filename='completion.png',
    width=1150,
)

## 5. Selective MEC and LEC degradation

These experiments isolate corruption of the cue-like LEC partition and the spatial MEC partition during frozen recall. The sparse CA3-like key is compared with a dense common-key control that deliberately collapses item-specific pre-activation. The expected qualitative result is a complementary impairment: LEC corruption preferentially harms cue identity, whereas MEC corruption harms position decoding while cue identity is relatively spared.

In [ ]:
lec_results = ROOT / 'results/mtl_cue_degradation/v1'
mec_results = ROOT / 'results/mtl_mec_degradation/v1'
degradation_plot = show_built_figure(
    build_degradation_figure,
    lec_results,
    mec_results,
    filename='selective_ec_degradation.png',
    width=1000,
)

### What selective degradation looks like

The examples below show one held-out lap at 50% modality-specific dropout. Rows correspond to LEC-only and MEC-only corruption. Columns show the intact target, degraded probe, recalled output using the sparse key, and recalled output using the dense common key. They are qualitative examples accompanying the seed-level curves above.

In [ ]:
lec_examples = load_arrays(lec_results)
mec_examples = load_arrays(mec_results)
example_sets = [('LEC-only dropout', lec_examples), ('MEC-only dropout', mec_examples)]
column_titles = ['Intact target', 'Degraded probe', 'Sparse-key output', 'Dense-key output']

figure, axes = plt.subplots(2, 4, figsize=(13, 6), constrained_layout=True)
for row, (row_label, arrays) in enumerate(example_sets):
    modes = arrays['key_modes'].tolist()
    values = [
        arrays['example_target'][0],
        arrays['example_probe'][0],
        arrays['example_output'][0, modes.index('normal')],
        arrays['example_output'][0, modes.index('dense')],
    ]
    for column, value in enumerate(values):
        image = axes[row, column].imshow(value.T, origin='lower', aspect='auto', cmap='viridis', vmin=0, vmax=1)
        axes[row, column].set(title=column_titles[column], xlabel='Track position')
        if column == 0:
            axes[row, column].set_ylabel(f'{row_label}\nEC/output unit')
figure.colorbar(image, ax=axes, fraction=0.018, pad=0.02, label='Activity')
figure.suptitle('Representative frozen recall at 50% selective input dropout', fontsize=14)
plt.show()

## 6. Direct versus error-driven plasticity

The supplementary control changes only the plasticity rule. Both rules inherit the same configuration, autoencoder, CA3 map, input sequence, seeds, and masks; they were not separately optimized. The bounded error-driven rule corrects under- and overexpressed CA1 components, whereas direct instructed writing does not use the current recall error.

In [ ]:
rule_plot = show_built_figure(
    build_rule_ablation,
    RESULTS / 'plasticity_ablation',
    filename='plasticity_rule_ablation.png',
    width=900,
)

## What the complete preprint claims—and what it does not

**Supported constructive claims**

- Rapid associative storage can preserve downstream decodability when instructed CA1 targets use an established representational basis.
- Cue-sequence changes can generate a continuous mixture of stable and context-dependent CA1 tuning in the model.
- Structured, item-specific CA3-like keys support more useful recall under input corruption than a degenerate common key.
- MEC-like and LEC-like input partitions make complementary contributions to spatial and cue recall.
- Error-driven and direct plasticity rules exhibit a clean-discrimination versus severe-corruption trade-off under matched parameters.

**Outside the present evidence**

- A quantitative fit to biological CA1 remapping statistics.
- Long-term representational stability across days or years.
- Recurrent CA3 pattern completion.
- A claim that backpropagation, Adam, or CMA-ES is implemented biologically.

## Regenerating the underlying simulations

The notebook only reads existing artifacts. To repeat a simulation, run the commands below from the repository root into a **new, empty output directory**; runners intentionally refuse to overwrite an existing artifact.

```bash
PYTHONPATH=src python3 -m experiments.preprint.compatibility \
  --config src/experiments/preprint/configs/final_compatibility.json \
  --output results/preprint/reproduction/compatibility

PYTHONPATH=src python3 -m experiments.preprint.cue_remapping \
  --config src/experiments/preprint/configs/final_cue_remapping.json \
  --output results/preprint/reproduction/cue_remapping

PYTHONPATH=src python3 -m experiments.preprint.cue_swap_control \
  --config src/experiments/preprint/configs/final_cue_remapping.json \
  --output results/preprint/reproduction/cue_swap_control

PYTHONPATH=src python3 -m experiments.preprint.completion \
  --config src/experiments/preprint/configs/final_completion.json \
  --output results/preprint/reproduction/completion

PYTHONPATH=src python3 -m experiments.preprint.plasticity_ablation \
  --config src/experiments/preprint/configs/final_plasticity_ablation.json \
  --output results/preprint/reproduction/plasticity_ablation
```

See `src/experiments/preprint/README.md` for the maintained command list and artifact conventions.